# Baseline

The scripted report the agent is compared with: plain Python, no model, a fixed recipe.
It reads a case's structured scope (`canonical_scope`, not the request text), calls the
same five tools through the same gateway, and writes the same kind of review packet.
It has no call limit.

The recipe:

1. If the scope leaves anything unspecified, ask (CLARIFY) for exactly that.
2. Read the context, compare the runs, and check the data over both stretches.
3. If the comparison is not fair (rows missing or duplicated, a different model
   version or different horizons on the two sides), stop (STOP_UNTRUSTWORTHY).
4. Otherwise answer (ANSWER): the error is worse if it rose by more than a tenth;
   when it is worse, look at the series day by day for the rows behind it.

A failed tool call is retried once. Causes are never claimed; they are listed as not
established. The baseline is frozen (git tag `baseline-frozen`) before any held-out case
exists, so it cannot be tuned to them.

In [ ]:
#| default_exp baseline

In [ ]:
#| export
from typing import Any

from agentic_forecast_review.gateway import (
    Gateway,
)
from agentic_forecast_review.generator import (
    Snapshot,
)
from agentic_forecast_review.schemas import (
    Case,
    Claim,
    ClaimLabel,
    Clarification,
    Cost,
    Metric,
    MissingScope,
    Packet,
    RunSetting,
    TerminalState,
    ToolResult,
    TraceStep,
)
from agentic_forecast_review.tools import (
    TOOLS,
)

## The rules as data

What makes a comparison unfair, as pairs of fields that must agree, and the threshold
for "worse".

In [ ]:
#| export
WORSE_IF_RISE_ABOVE = 0.10  # a rise in MAE of more than a tenth of the window's MAE
NOT_ESTABLISHED = "the cause of any change in the error; no tool measures causes"

## One scripted run

`Script` records each call as a trace step with a fixed summary and the earlier calls
it relies on, exactly as the agent's steps are recorded.

In [ ]:
#| export
class Script:
    def __init__(
        self, case: Case, snapshot: Snapshot, setting: RunSetting | None
    ) -> None:
        self.case = case
        self.gateway = Gateway(case, snapshot, TOOLS, setting)
        self.steps: list[TraceStep] = []

    def call(
        self, request: dict[str, Any], summary: str, refs: tuple[str, ...] = ()
    ) -> ToolResult:
        """Send a request; retry once if it fails; record every attempt."""
        for _ in range(2):
            reply = self.gateway.call(request)
            self.steps.append(
                TraceStep(
                    turn=len(self.steps) + 1,
                    summary=summary,
                    observation_refs=refs,
                    call=reply.call,
                    result=reply.result if reply.call is not None else None,
                )
            )
            if reply.result.ok:
                return reply.result
        raise RuntimeError(f"{request['tool']} failed twice: {reply.result.error}")

    def packet(
        self,
        state: TerminalState,
        claims: tuple[Claim, ...] = (),
        clarification: Clarification | None = None,
        summary: str = "finish",
        refs: tuple[str, ...] = (),
    ) -> Packet:
        trace = (
            *self.steps,
            TraceStep(turn=len(self.steps) + 1, summary=summary, observation_refs=refs),
        )
        return Packet(
            case_id=self.case.case_id,
            runner="baseline",
            snapshot_id=self.case.snapshot_id,
            terminal_state=state,
            claims=claims,
            clarification=clarification,
            not_established=(NOT_ESTABLISHED,) if state is TerminalState.ANSWER else (),
            trace=trace,
            cost=Cost(
                turns=len(trace),
                calls=sum(step.call is not None for step in trace),
                usd=0,
                seconds=0,
            ),
        )

## The recipe

Unfair comparisons become STOP claims that cite the tool fields showing them: FINDINGs
where `check_quality` names the affected rows, INFERENCEs otherwise.

In [ ]:
#| export
def _token(call_id: str, field: str) -> str:
    return "{ev:" + call_id + "." + field + "}"


def _unfair(series_id: str, cmp: ToolResult, dq: ToolResult) -> list[Claim]:
    """Claims for every way the comparison of one series is not fair."""
    p, c, d = series_id.lower() + "__", cmp.call_id, dq.call_id
    f, q = cmp.fields, dq.fields
    claims: list[Claim] = []
    rows = tuple(row for row in dq.row_ids if f"__{series_id}__" in row)
    if q[p + "missing_days"] or q[p + "duplicate_rows"]:
        claims.append(
            Claim(
                label=ClaimLabel.FINDING,
                template=f"{series_id} has {_token(d, p + 'missing_days')} missing actual "
                f"days and {_token(d, p + 'duplicate_rows')} duplicated actual rows across "
                "the two stretches.",
                row_refs=rows,
            )
        )
    for side in ("period", "window"):
        if f[p + f"n_actual_{side}"] != f[p + f"n_expected_{side}"]:
            claims.append(
                Claim(
                    label=ClaimLabel.INFERENCE,
                    template=f"The {side} error of {series_id} rests on "
                    f"{_token(c, p + f'n_actual_{side}')} actual rows for "
                    f"{_token(c, p + f'n_expected_{side}')} days, so it is not comparable.",
                )
            )
    if f[p + "model_window"] != f[p + "model_period"]:
        claims.append(
            Claim(
                label=ClaimLabel.INFERENCE,
                template=f"For {series_id} the window was forecast by "
                f"{_token(c, p + 'model_window')} and the period by "
                f"{_token(c, p + 'model_period')}, so the error change mixes two models.",
            )
        )
    if f[p + "horizons_window"] != f[p + "horizons_period"]:
        claims.append(
            Claim(
                label=ClaimLabel.INFERENCE,
                template=f"For {series_id} the window was judged at horizons "
                f"{_token(c, p + 'horizons_window')} and the period at "
                f"{_token(c, p + 'horizons_period')}, so the errors are not comparable.",
            )
        )
    return claims


def run_baseline(
    case: Case, snapshot: Snapshot, setting: RunSetting | None = None
) -> Packet:
    """The scripted review of one case."""
    script = Script(case, snapshot, setting)
    scope = case.canonical_scope
    missing = scope.missing()
    if missing:
        options = (
            {MissingScope.METRIC: tuple(metric.value for metric in Metric)}
            if MissingScope.METRIC in missing
            else {}
        )
        return script.packet(
            TerminalState.CLARIFY,
            clarification=Clarification(missing=missing, options=options),
            summary="the request leaves part of its scope unspecified",
        )
    assert scope.metric and scope.period and scope.window and scope.series
    period, window = scope.period, scope.window
    ctx = script.call({"tool": "resolve_context"}, "read the snapshot context")
    cmp = script.call(
        {
            "tool": "compare_runs",
            "series": list(scope.series),
            "metric": scope.metric.value,
            "period": period.model_dump(),
            "window": window.model_dump(),
        },
        "compare the review period with the window",
        (ctx.call_id,),
    )
    both = {"start_day": window.start_day, "end_day": period.end_day}
    dq = script.call(
        {"tool": "check_quality", "series": list(scope.series), "days": both},
        "check the data behind both stretches",
        (cmp.call_id,),
    )
    unfair = [
        claim for series_id in scope.series for claim in _unfair(series_id, cmp, dq)
    ]
    if unfair:
        return script.packet(
            TerminalState.STOP_UNTRUSTWORTHY,
            claims=tuple(unfair),
            summary="the comparison is not fair as computed",
            refs=(cmp.call_id, dq.call_id),
        )

    claims: list[Claim] = []
    for series_id in scope.series:
        p, c = series_id.lower() + "__", cmp.call_id
        rise = float(cmp.fields[p + "mae_delta"])
        worse = rise > WORSE_IF_RISE_ABOVE * float(cmp.fields[p + "mae_window"])
        change = (
            f"{series_id}: the error rose from {_token(c, p + 'mae_window')} to "
            f"{_token(c, p + 'mae_period')} on a fair comparison, and the bias moved from "
            f"{_token(c, p + 'bias_window')} to {_token(c, p + 'bias_period')}."
            if worse
            else f"{series_id}: the error did not get materially worse "
            f"({_token(c, p + 'mae_window')} to {_token(c, p + 'mae_period')})."
        )
        if worse:
            ins = script.call(
                {
                    "tool": "inspect_series",
                    "series_id": series_id,
                    "days": period.model_dump(),
                },
                "look at the worse period day by day",
                (cmp.call_id,),
            )
            claims.append(
                Claim(label=ClaimLabel.FINDING, template=change, row_refs=ins.row_ids)
            )
        else:
            claims.append(Claim(label=ClaimLabel.INFERENCE, template=change))
    return script.packet(
        TerminalState.ANSWER,
        claims=tuple(claims),
        summary="the comparison is fair; report the change",
        refs=(cmp.call_id, dq.call_id),
    )

## The nine development cases

The baseline on every dev case, and its ending against the answer key (the grader's
view; the baseline itself never reads the manifest). It was written knowing the dev
conditions, so a clean sweep here says little; the held-out cases are the real test.

In [ ]:
from agentic_forecast_review.cases import build_family
from agentic_forecast_review.validator import known_identifiers, packet_stray_digits

family = build_family()
settings = {setting.case_id: setting for setting in family.run_settings}
correct = 0
for case, entry in zip(family.cases, family.manifest, strict=True):
    snapshot = family.snapshots[case.snapshot_id]
    packet = run_baseline(case, snapshot, settings.get(case.case_id))
    assert packet_stray_digits(packet, known_identifiers(case, snapshot, packet)) == {}
    ok = packet.terminal_state is entry.expected_terminal
    correct += ok
    print(
        f"{case.case_id}: {packet.terminal_state.value:18} calls {packet.cost.calls}  {'ok' if ok else 'WRONG'}"
    )
    for line in packet.render():
        print("    -", line)
print(f"{correct} of {len(family.cases)} endings match the answer key")
assert correct == len(family.cases)